# Hotel Booking Cancellation Prediction and Booking Segmentation

Group 5

HE CAI · LI YUECHAO · LIANG SIMAN · NIU JINGQI · YOU YIRAN

This notebook is the common reading and verification entry point. The report integrates data preparation, EDA, clustering, four classifiers, forest explanations and probability assessment. Reproduction commands write to a separate run directory.

In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
assert (project_root / 'data/hotel_bookings_clean.csv').is_file(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(project_root / 'src'))
import pandas as pd
from preprocessing_source import load_shared_data
data=load_shared_data(project_root)
print({"bookings":len(data["df"]),"inputs":len(data["features"]),"random_train":len(data["random_train"]),"random_test":len(data["random_test"])})

## A Data and common protocol

118,564 cleaned bookings; cancellation=1; 24 supervised inputs; random seed 42; fixed five training folds. Cleaning is documented in A_02 and the final preprocessing contract in A_04. Later-arrival refits selected with random folds must be distinguished from the separate chronological forest selection.

In [ ]:
display(pd.read_csv(project_root/'data/data_dictionary.csv'))

## B Descriptive analysis and segmentation

K-means is fitted on five numeric booking attributes in random training only. Cancellation is used for post-fit profiling only. ADR is a descriptive clustering input and is excluded from the supervised classifiers.

In [ ]:
display(pd.read_csv(project_root/'results/tables/cluster_profile_full.csv'))
display(pd.read_csv(project_root/'results/tables/cluster_rates_by_split.csv'))

## C D E Common model comparison

All main decisions use threshold 0.50 and AP means average precision, not trapezoidal PR area. The separate forest time check is not a fair time comparison of all algorithms.

In [ ]:
display(pd.read_csv(project_root/'results/tables/integrated_model_comparison.csv'))

## E Calibration and thresholds

Training-derived out-of-fold predictions fit the calibration maps and choose thresholds. Test data evaluate those fixed choices. Results do not establish prospective policy value.

In [ ]:
display(pd.read_csv(project_root/'results/tables/integrated_probability_assessment.csv'))

## D Forest interpretation

These SHAP values explain the saved random-training forest. They do not explain XGBoost or estimate causal effects.

In [ ]:
display(pd.read_csv(project_root/'results/tables/random_shap_importance.csv').head(10))

## Verification

The following rechecks saved prediction alignment, folds and metrics. It does not repeat every hyperparameter search.

In [ ]:
from verification import verify
verification=verify(project_root,check_models=False)
assert verification["status"]=="PASS"
print(verification["status"])

## Reproduce in a separate directory

From the project root:

```bash
python src/run_project.py --stage verify --check-models
python src/run_project.py --stage ab --output-root runs/reproduction
python src/run_project.py --stage dummy --output-root runs/reproduction
python src/run_project.py --stage logistic --output-root runs/reproduction
python src/run_project.py --stage forest --output-root runs/reproduction
python src/run_project.py --stage explain --output-root runs/reproduction
```

The README gives the XGBoost and calibration command for the completed E experiment. Reproduction can take substantial time. The submitted prediction CSVs remain the portable reference evidence.